## Statistical Post Processing Model:
1. Read the fully diacritized training corpus and split it into individual word tokens.
2. For each word, remove diacritics to obtain its undiacritized (base) form.
3. Build a dictionary mapping each undiacritized word to all diacritized forms observed in training.
4. Count how frequently each diacritized word appears to create a unigram frequency table.
5. Run the neural diacritization model to produce a diacritized sentence.
6. Split the neural output into words and process each word independently.
7. Remove diacritics from the predicted word to obtain its base form.
8. If the base form exists in the training dictionary but the predicted form does not, mark it as a correction candidate.
9. Replace the predicted word with the most frequent diacritized form observed in training for that base form.
10. Leave the word unchanged if it was already observed in training or the base form is unknown.
11. Recombine the corrected words to produce the final post-processed diacritized sentence.

In [5]:
import unicodedata
from collections import Counter, defaultdict
from typing import List

def is_combining(ch: str) -> bool:
    # True for Arabic tashkeel & other combining marks
    return unicodedata.combining(ch) != 0

def strip_diacritics(word: str) -> str:
    # remove all combining marks
    return ''.join(ch for ch in word if not is_combining(ch))

def strip_diacritics_sentence(sent: str) -> str:
    return " ".join(strip_diacritics(w) for w in sent.split())

In [ ]:
train_path = "dataset/train.txt"  

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_lines = read_lines("dataset/train.txt")
val_lines   = read_lines("dataset/val.txt")  

print("Train sentences:", len(train_lines))
print("Val sentences:  ", len(val_lines))

Train sentences: 50000
Val sentences:   2500


In [ ]:
# build lexicon and unigram frequencies from train.txt

def build_lexicon_and_unigrams(lines: List[str]):
    """
    Build:
      - lexicon: base (undiacritized) word -> Counter of diacritized forms
      - word_freq: unigram counts of diacritized words
    """
    lexicon = defaultdict(Counter)
    word_freq = Counter()
    
    for line in lines:
        for word in line.split():
            # count the fully diacritized word itself
            word_freq[word] += 1
            
            # map base form to diacritized variant
            base = strip_diacritics(word)
            if base: 
                lexicon[base][word] += 1
                
    return lexicon, word_freq

lexicon, word_freq = build_lexicon_and_unigrams(train_lines)

print(f"Number of base forms in lexicon: {len(lexicon)}")
some_base = next(iter(lexicon))
print("Example base form:", some_base)
print("Top variants:", lexicon[some_base].most_common(5))

# Input 
# lines = 
# [
#     "كَتَبَ الطَّالِبُ الدَّرسَ",
#     "قَرَأَ الطَّالِبُ الكِتابَ",
# ]

# OutPut
# lexicon
# {
#     "كتب": Counter(
#     {
#         "كَتَبَ": 5,
#         "كُتُبٌ": 2
#     }),
#     "الطالب": Counter(
#     {
#         "الطَّالِبُ": 8,
#         "الطَّالِبَ": 3,
#         "الطَّالِبِ": 1
#     })
# }

# word_freq
# {
#     "كَتَبَ": 5,
#     "كُتُبٌ": 2,
#     "الطَّالِبُ": 8,
#     "الطَّالِبَ": 3,
#     "الطَّالِبِ": 1
# }

Number of base forms in lexicon: 116078
Example base form: ولو
Top variants: [('وَلَوْ', 8551), ('وَلَوِ', 1)]


In [ ]:
import pickle

with open("lexicon.pkl", "wb") as f:
    pickle.dump(lexicon, f)

In [ ]:
# unigram post-processing

def postprocess_word(predicted_word: str, lexicon) -> str:
    """
    Apply unigram post-processing to a single predicted word.
    - If base form was seen in training and predicted form wasn't,
        replace with most frequent diacritized form for that base.
    """
    base = strip_diacritics(predicted_word)
    
    # base never seen in training → cannot correct
    if not base or base not in lexicon:
        return predicted_word
    
    candidates = lexicon[base]
    
    # predicted form already observed in training → trust the model
    if predicted_word in candidates:
        return predicted_word
    
    # otherwise replace with the most frequent diacritized form
    most_frequent_form = candidates.most_common(1)[0][0]
    return most_frequent_form

def postprocess_sentence(predicted_sentence: str, lexicon) -> str:
    """Apply unigram post-processing to every word in a predicted sentence."""
    words = predicted_sentence.split()
    corrected_words = [postprocess_word(w, lexicon) for w in words]
    return " ".join(corrected_words)


# Example
# Assume training data contained:
# lexicon["كتب"] =
# {
#     "كَتَبَ": 12,
#     "كُتُبٌ": 3
# }

# Case 1 — valid prediction
# postprocess_word("كَتَبَ", lexicon)
# → "كَتَبَ"      already seen, keep it

# Case 2 — invalid/unseen prediction
# postprocess_word("كُتِبَ", lexicon)
# → "كَتَبَ"      replace with most frequent form

# Case 3 — unseen word
# postprocess_word("مُعَالِجَةٌ", lexicon)
# → "مُعَالِجَةٌ"    unchanged

In [10]:
def predict_with_unigram_baseline(gold_lines, lexicon):
    preds = []
    for gold in gold_lines:
        undiac = strip_diacritics_sentence(gold)
        pred   = postprocess_sentence(undiac, lexicon)
        preds.append(pred)
    return preds

val_pred_lines = predict_with_unigram_baseline(val_lines, lexicon)

print("Example:")
print("GOLD:", val_lines[0])
print("UNDIAC:", strip_diacritics_sentence(val_lines[0]))
print("PRED:", val_pred_lines[0])

Example:
GOLD: الشَّهَادَةِ ظَاهِرَةً ، وَبِحَقٍّ بَيِّنٍ تَضْعُفُ التُّهْمَةُ ، وَهُوَ الْفَرْقُ بَيْنَهُ وَبَيْنَ الشَّهَادَةِ ، وَعَنْ أَصْبَغَ الْجَوَازُ فِي الْوَلَدِ وَالزَّوْجَةِ وَالْأَخِ وَالْمُكَاتَبِ وَالْمُدَبَّرِ وَالْمِدْيَانِ إنْ كَانَ مِنْ أَهْلِ الْقِيَامِ بِالْحَقِّ ، وَصَحَّ الْحُكْمُ ، وَقَدْ يَحْكُمُ لِلْخَلِيفَةِ ، وَهُوَ فَوْقَهُ ، وَتُهْمَتُهُ أَقْوَى ، وَلَا يَنْبَغِي لَهُ الْقَضَاءُ بَيْنَ أَحَدٍ مِنْ عَشِيرَتِهِ وَخَصْمِهِ ، وَإِنْ رَضِيَ الْخَصْمُ بِخِلَافِ رَجُلَيْنِ رَضِيَا بِحُكْمِ رَجُلٍ أَجْنَبِيٍّ فَيَنْفُذُ ذَلِكَ عَلَيْهِمَا ، وَلَا يَقْضِي بَيْنَهُ وَبَيْنَ غَيْرِهِ ، وَإِنْ رَضِيَ الْخَصْمُ بِذَلِكَ فَإِنْ فَعَلَ فَيُشْهِدُ عَلَى رِضَاهُ ، وَيَجْتَهِدُ فِي الْحَقِّ فَإِنْ قَضَى لِنَفْسِهِ أَوْ لِمَنْ يَمْتَنِعُ قَضَاؤُهُ لَهُ فَلْيَذْكُرْ الْقِصَّةَ كُلَّهَا ، وَرَضِيَ خَصْمِهِ ، وَشَهَادَةَ مَنْ شَهِدَ بِرِضَى الْخَصْمِ .
UNDIAC: الشهادة ظاهرة ، وبحق بين تضعف التهمة ، وهو الفرق بينه وبين الشهادة ، وعن أصبغ الجواز في الولد والزوجة والأخ والمكاتب وا

In [11]:
def word_accuracy(gold_lines, pred_lines):
    assert len(gold_lines) == len(pred_lines)
    correct = 0
    total = 0
    for g, p in zip(gold_lines, pred_lines):
        g_words = g.split()
        p_words = p.split()
        # if lengths mismatch, compare up to min length
        for gw, pw in zip(g_words, p_words):
            total += 1
            if gw == pw:
                correct += 1
    return correct / total if total else 0.0

def char_accuracy(gold_lines, pred_lines):
    correct = 0
    total = 0
    for g, p in zip(gold_lines, pred_lines):
        for gc, pc in zip(g, p):
            total += 1
            if gc == pc:
                correct += 1
    return correct / total if total else 0.0

w_acc  = word_accuracy(val_lines, val_pred_lines)
c_acc  = char_accuracy(val_lines, val_pred_lines)

print(f"Word accuracy: {w_acc:.4f}")
print(f"Char accuracy: {c_acc:.4f}")

Word accuracy: 0.7071
Char accuracy: 0.1729
